In [ ]:
% matplotlib inline

% load_ext autoreload
% autoreload 2

In [ ]:
import sys
from os.path import exists

sys.path.append('../..')

In [ ]:
import numpy as np
from loguru import logger

from stable_baselines3.common.env_checker import check_env

In [ ]:
from vimms.Common import POSITIVE, load_obj, save_obj
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler, \
    MZMLChromatogramSampler
from vimms.Roi import RoiBuilderParams

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.evaluation import run_method

# 1. Parameters

In [ ]:
n_chemicals = (5000, 20000)
mz_range = (70, 1000)
rt_range = (0, 1440)
intensity_range = (1E4, 1E20)

In [ ]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [ ]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1E3

In [ ]:

mzml_filename = '../fullscan_QCB.mzML'
samplers = None
samplers_pickle = 'samplers_%s.p' % mzml_filename
if exists(samplers_pickle):
    logger.info('Loaded %s' % samplers_pickle)
    samplers = load_obj(samplers_pickle)
    mz_sampler = samplers['mz']
    ri_sampler = samplers['rt_intensity']
    cr_sampler = samplers['chromatogram']
else:
    logger.info('Creating samplers from %s' % mzml_filename)
    mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
    ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                           min_log_intensity=min_log_intensity,
                                           max_log_intensity=max_log_intensity)
    roi_params = RoiBuilderParams(min_roi_length=3, at_least_one_point_above=1000)
    cr_sampler = MZMLChromatogramSampler(mzml_filename, roi_params=roi_params)
    samplers = {
        'mz': mz_sampler,
        'rt_intensity': ri_sampler,
        'chromatogram': cr_sampler
    }
    save_obj(samplers, samplers_pickle)

In [ ]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

### Test environment

In [ ]:
max_peaks = 100
env = DDAEnv(max_peaks, params)
check_env(env)

In [ ]:
n_eval_episodes = 1
out_dir = 'synthetic_data_check'
methods = [
    'TopN',
    'random',
]

In [ ]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    chems = generate_chemicals(chemical_creator_params)
    print(len(chems))
    chem_list.append(chems)

In [ ]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()

    model = None
    env = DDAEnv(max_peaks, copy_params)
    run_method(env, chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model,
               N=10, print_eval=True)
    print()